Implementing Vocabulary to Random Forest

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Scikit-learn tools for modeling and evaluation
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Text processing tools
import re
from collections import Counter
from nltk.corpus import stopwords, names
import nltk

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('names')

# Load dataset
df = pd.read_csv("/content/test.csv")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [ ]:
# Filter rows labeled as individuals (label = 1)
person_rows = df[df['label'] == 1]

# Tokenize full names into lowercase word tokens
person_tokens = person_rows['full_name'].dropna().apply(lambda x: re.findall(r'\b\w+\b', x.lower()))
flat_person_names = [name for sublist in person_tokens for name in sublist]

# Initialize filtering criteria
stop_words = set(stopwords.words('english'))
unwanted_titles = {'mr', 'mrs', 'ms', 'dr', 'jr', 'sr', 'miss', 'rev'}
name_list = set(name.lower() for name in names.words())

# Clean person name tokens by removing stopwords, titles
cleaned_person_names = [
    name for name in flat_person_names
    if name not in stop_words
    and name not in unwanted_titles
    and len(name) > 1
    and name in name_list
]

# Get top 10 most common cleaned person names
top_names = [name for name, count in Counter(cleaned_person_names).most_common(10)]
print("Top 10 individual names:", top_names)

Top 10 individual names: ['john', 'michael', 'robert', 'david', 'james', 'richard', 'thomas', 'william', 'mary', 'mark']


In [ ]:
# Filter rows labeled as organizations (label = 0)
org_rows = df[df['label'] == 0]

# Tokenize full names into lowercase word tokens
org_tokens = org_rows['full_name'].dropna().apply(lambda x: re.findall(r'\b\w+\b', x.lower()))
flat_org_words = [word for sublist in org_tokens for word in sublist]

# Clean organization word tokens by removing stopwords, titles, person names, and short tokens
cleaned_org_words = [
    word for word in flat_org_words
    if word not in stop_words
    and word not in unwanted_titles
    and word not in top_names
    and len(word) > 1
]

# Get top 10 most common cleaned organization-related words
top_org_words = [word for word, count in Counter(cleaned_org_words).most_common(10)]
print("Top 10 org-related words:", top_org_words)

Top 10 org-related words: ['committee', 'elect', 'pac', 'inc', 'friends', 'arizona', 'campaign', 'state', 'llc', 'senate']


In [ ]:
def is_pac(full_name, pac_words):
    '''
    Check if the given full name contains any token from the specified PAC-related words.
    Args:
        full_name (str): The full name string to analyze.
        pac_words (list or set): Collection of PAC-related tokens to match against.
    Returns:
        int: 1 if any token in full_name is found in pac_words, otherwise 0.
    '''
    if not isinstance(full_name, str):
        return 0
    name_list = re.findall(r'\b\w+\b', full_name.lower())
    return int(any(name in pac_words for name in name_list))


def is_org(full_name, org_words):
    '''
    Check if the given full name contains any token from the specified organization-related words.
    Args:
        full_name (str): The full name string to analyze.
        org_words (list or set): Collection of organization-related tokens to match against.
    Returns:
        int: 1 if any token in full_name is found in org_words, otherwise 0.
    '''
    if not isinstance(full_name, str):
        return 0
    name_list = re.findall(r'\b\w+\b', full_name.lower())

In [ ]:
# Split data into training and validation sets (75% train, 25% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=1)

In [ ]:
# Initialize and train Random Forest classifier
rfc = RandomForestClassifier(random_state=1)
rfc.fit(X_train, y_train)

# Predict labels for validation set
predicted = rfc.predict(X_val)

ValueError: could not convert string to float: 'discessio, l.l.c'

In [ ]:
# Calculate evaluation metrics and convert to percentage
accuracy = accuracy_score(y_val, predicted) * 100
recall = recall_score(y_val, predicted) * 100
precision = precision_score(y_val, predicted) * 100
f1 = f1_score(y_val, predicted) * 100

# Print the results
print("Accuracy:", accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("f1_score:", f1)
print()

# Display detailed classification report
print(classification_report(y_val, predicted))

In [ ]:
# Compute confusion matrix for validation predictions
cm = confusion_matrix(y_val, predicted, labels=[0, 1])

# Visualize confusion matrix with custom class labels
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Individual', 'Organization'])
disp.plot(cmap=plt.cm.Blues)

plt.title("Flagged Random Forest Confusion Matrix - pa annotated")
plt.show()